In [2]:
import pandas as pd

customers = pd.read_csv("../data/raw/customers.csv")
subscriptions = pd.read_csv("../data/raw/subscriptions.csv")
revenue = pd.read_csv("../data/raw/revenue.csv")

In [3]:
customers['signup_date'] = pd.to_datetime(customers['signup_date'])
customers['churn_date'] = pd.to_datetime(customers['churn_date'])

subscriptions['month'] = pd.to_datetime(subscriptions['month'])
revenue['month'] = pd.to_datetime(revenue['month'])


In [4]:
#First subscription month per cusomer
first_subscription =(
    subscriptions
    .groupby('customer_id')['month']
    .min()
    .reset_index(name ='first_subscription_month')
)
activation = customers.merge(
    first_subscription,
    on='customer_id',
    how='left'
)
activation['days_to_activate']=(
    activation['first_subscription_month'] - activation['signup_date']
).dt.days


In [5]:
# Convert each customer's signup date into a monthly period (e.g., 2023-05)
# This allows us to compare dates at the month level instead of exact timestamps.
customers['signup_month'] = customers['signup_date'].dt.to_period('M')


# If a customer has not churned, churn_month will be NaT (Not a Time).
customers['churn_month'] = customers['churn_date'].dt.to_period('M')

# Calculate the customer's lifetime in months.
# Subtracting two Period('M') objects gives a Period difference.
# The `.n` attribute extracts the numeric number of months from that difference.
# If churn_month is missing (customer still active), return None instead.
customers['lifetime_months'] = (
    customers['churn_month'] - customers['signup_month']
).apply(lambda x: x.n if pd.notnull(x) else None)

In [6]:
customers['lifetime_months'].describe()


count    168.000000
mean       5.315476
std        4.078775
min        0.000000
25%        2.000000
50%        4.000000
75%        8.000000
max       17.000000
Name: lifetime_months, dtype: float64

In [7]:
customers['lifetime_months'].value_counts().sort_index()


lifetime_months
0.0      7
1.0     15
2.0     29
3.0     26
4.0     13
5.0     14
6.0     10
7.0     10
8.0     10
9.0      7
10.0     2
11.0     5
12.0     6
13.0     4
14.0     5
15.0     3
16.0     1
17.0     1
Name: count, dtype: int64

In [8]:
# Build a retention-style table based on customer lifetimes.

retention = (
    customers
        # Remove customers who do not have a lifetime_months value.
        # These are typically active customers who haven't churned yet.
        .dropna(subset=['lifetime_months'])

        # Group customers by how many months they stayed before churning.
        # Example: all customers who churned after 3 months go into the same group.
        .groupby('lifetime_months')

        # Count how many customers are in each lifetime group.
        # This gives the number of customers who churned at each month.
        .size()

        # Convert the counts into a cumulative sum.
        # This shows how many customers have churned *up to* each lifetime month.
        # Example: if 5 churned at month 1 and 7 at month 2, cumsum = [5, 12].
        .cumsum()

        # Turn the result into a clean DataFrame with a readable column name.
        .reset_index(name='churned_customers')
)

retention

,lifetime_months,churned_customers
0,0.0,7
1,1.0,22
2,2.0,51
3,3.0,77
4,4.0,90
5,5.0,104
6,6.0,114
7,7.0,124
8,8.0,134
9,9.0,141


## Retention & Churn Insight
Customer churn is heavily front-loaded, with a median lifetime of approximately 4 months. Nearly half of all churned customers leave within the first three months, indicating that early post-activation experience is the primary driver of retention outcomes.


In [9]:
customers['cohort_month'] = customers['signup_date'].dt.to_period('M')


In [10]:
# Convert the subscription month column into a monthly Period type.
# This standardizes all dates to the month level (e.g., 2023-05),
# which is essential for cohort analysis and retention tracking.
subscriptions['month'] = subscriptions['month'].dt.to_period('M')

# Build a table that contains one row per customer per month they were active.
customer_months = (
    subscriptions[['customer_id', 'month']]
        # Remove duplicate rows in case the same customer-month appears multiple times.
        # This ensures each customer is counted only once per month.
        .drop_duplicates()

        # Attach each customer's cohort_month (the month they first signed up).
        # This allows us to compare activity months to the customer's cohort.
        .merge(
            customers[['customer_id', 'cohort_month']],
            on='customer_id',
            how='left'   # Keep all customer-month rows even if cohort_month is missing.
        )
)

customer_months

,customer_id,month,cohort_month
0,1020,2024-10,2024-10
1,1020,2024-11,2024-10
2,1020,2024-12,2024-10
3,1020,2025-01,2024-10
4,1020,2025-02,2024-10
...,...,...,...
983,1988,2025-01,2024-11
984,1988,2025-02,2024-11
985,1988,2025-03,2024-11
986,1999,2025-04,2025-04


In [11]:
# Calculate how many months have passed since each customer signed up.
# Both 'month' and 'cohort_month' are Period('M') objects.
# Subtracting them gives a Period difference (e.g., <3 * Month>).
# The `.n` attribute extracts the numeric value of that difference.
# This produces an integer representing the customer's "age" in months
# relative to their cohort start date.
customer_months['months_since_signup'] = (
    customer_months['month'] - customer_months['cohort_month']
).apply(lambda x: x.n)

In [14]:
cohort_retention = (
    customer_months
    .groupby(['cohort_month', 'months_since_signup'])
    .agg(active_users=('customer_id', 'nunique'))
    .reset_index()
)
cohort_retention.head()

,cohort_month,months_since_signup,active_users
0,2024-01,0,10
1,2024-01,1,9
2,2024-01,2,9
3,2024-01,3,9
4,2024-01,4,9


In [17]:
# Calculate the size of each cohort.
# We group by cohort_month and take the FIRST value of active_users.
# Why .first()? Because in a cohort retention table, the first row for each cohort
# represents month 0 (the signup month), which contains the total number of users
# who joined that cohort.
cohort_sizes = (
    cohort_retention
        .groupby('cohort_month')['active_users']
        .first()
        .reset_index(name='cohort_size')  # Rename the column for clarity
)

# Merge the cohort sizes back into the main retention table.
# This gives every row (each cohort-month combination) access to its cohort size.
cohort_retention = cohort_retention.merge(
    cohort_sizes,
    on='cohort_month'
)

# Compute the retention rate for each cohort-month.
# Retention rate = active users in that month / total users in the cohort.
# Example: If a cohort had 100 users at signup and 60 are active in month 2,
# retention_rate = 0.60 (60%).

cohort_retention['retention_rate'] = (
    cohort_retention['active_users'] /
    cohort_retention['cohort_size']
)
cohort_retention.head()

,cohort_month,months_since_signup,active_users,cohort_size_x,retention_rate,cohort_size_y,cohort_size
0,2024-01,0,10,10,1.0,10,10
1,2024-01,1,9,10,0.9,10,10
2,2024-01,2,9,10,0.9,10,10
3,2024-01,3,9,10,0.9,10,10
4,2024-01,4,9,10,0.9,10,10


In [18]:
retention_matrix = cohort_retention.pivot(
    index='cohort_month',
    columns='months_since_signup',
    values='retention_rate'
)

retention_matrix


months_since_signup,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16
cohort_month,,,,,,,,,,,,,,,,,
2024-01,1.0,0.9,0.900000,0.900000,0.900000,0.700000,0.700000,0.700000,0.700000,0.700000,0.600000,0.500000,0.400000,0.300000,0.200000,0.100000,0.1
2024-02,1.0,1.0,0.875000,0.875000,0.875000,0.750000,0.750000,0.750000,0.750000,0.625000,0.625000,0.625000,0.625000,0.500000,0.125000,0.125000,NaN
2024-03,1.0,1.0,1.000000,0.846154,0.692308,0.692308,0.692308,0.615385,0.538462,0.461538,0.461538,0.461538,0.461538,0.461538,0.384615,0.153846,NaN
2024-04,1.0,1.0,0.800000,0.800000,0.700000,0.700000,0.700000,0.600000,0.600000,0.500000,0.400000,0.400000,0.200000,NaN,NaN,NaN,NaN
2024-05,1.0,1.0,1.000000,0.800000,0.600000,0.600000,0.500000,0.500000,0.400000,0.400000,0.400000,0.300000,0.200000,NaN,NaN,NaN,NaN
2024-06,1.0,1.0,1.000000,0.900000,0.900000,0.700000,0.500000,0.500000,0.500000,0.300000,0.100000,0.100000,NaN,NaN,NaN,NaN,NaN
2024-07,1.0,1.0,0.600000,0.500000,0.500000,0.500000,0.500000,0.500000,0.400000,0.200000,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2024-08,1.0,1.0,0.875000,0.750000,0.625000,0.500000,0.500000,0.250000,0.125000,0.125000,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2024-09,1.0,1.0,0.833333,0.500000,0.500000,0.500000,0.333333,0.166667,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
